# AI Bootcamp — Section 5: How Agents Talk
## Practical 2 · **Strands Agent + MCP Tools + Ollama (no AWS)**

**What you will build:** A real **MCP server** in a local Python subprocess, an **MCP client** that discovers and calls its tools, and a **Strands AI Agent** connected to your existing `qwen3:1.7b` local Ollama model.

All inventory, SKU, pricing and shipping values are **fictional training data**. No transactions, logins, cloud services, AWS keys, or paid model APIs are involved. Open-source packages may need a one-time internet download.

**Windows/Jupyter fix:** MCP stderr is redirected to a normal file (`section5_mcp_stderr.log`) because ipykernel stderr does not provide a valid Windows `fileno()`.

**Run code cells 1 → 8 in order, using `Shift + Enter`.** JupyterLab normally supports top-level `await`. If a cell fails, send a screenshot of its **last** error line and do not mark the entire lab complete yet.

**Important distinction:** In Cell 5, *Python* explicitly invokes MCP tools. In Cell 7, the *Strands LLM Agent* is asked to select the MCP tools. We record actual server calls to confirm whether it did so; a small local model may not reliably choose tools.

### Cell 1 — Install / verify the local Strands + MCP libraries

Run this **in JupyterLab**, not PowerShell. This installs libraries into the active Jupyter kernel. After a major upgrade you may need **Kernel → Restart Kernel**, then rerun from Cell 1.

In [ ]:
# Cell 1 — Open-source packages; no paid APIs
import sys, subprocess, importlib.metadata as metadata
packages = ["strands-agents[ollama]", "mcp>=2,<3"]
print("Active Python:", sys.executable)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])
print("Strands:", metadata.version("strands-agents"))
print("MCP SDK:", metadata.version("mcp"))
print("PASS: Strands and MCP dependencies installed")

### Cell 2 — Check Ollama and Qwen3

This checks the local Ollama API. Start Ollama if it isn't running. No LLM generation happens in this step.

In [ ]:
# Cell 2 — Local model and imports
import json
from urllib.request import urlopen
from pathlib import Path

from strands import Agent
from strands.models.ollama import OllamaModel
from strands.tools.mcp import MCPClient
from mcp import stdio_client, StdioServerParameters, ClientSession

OLLAMA_HOST = "http://127.0.0.1:11434"
OLLAMA_MODEL = "qwen3:1.7b"
with urlopen(OLLAMA_HOST + "/api/tags", timeout=10) as response:
    names = [m.get("name", "") for m in json.load(response).get("models", [])]
assert OLLAMA_MODEL in names, f"Download model using: ollama pull {OLLAMA_MODEL}. Found: {names}"
print("PASS: Ollama running and Qwen3 installed")
print("PASS: Strands Agent + MCPClient + OllamaModel imports")

### Cell 3 — Create a real local MCP server

The code below writes `mcp_shop_server.py` beside this notebook. The server implements **two tools**, **one read-only resource** and **one prompt**. No separate terminal is needed; MCP will start the server in a subprocess.

In [ ]:
# Cell 3 — Write a real MCP stdio server to disk
from pathlib import Path
import ast
SERVER_SOURCE = '# Section 5 Practical 2 — fictional shop MCP server.\n# NOTE: MCP stdio uses stdout for protocol messages. Do not print from the server.\nfrom pathlib import Path\nimport json\nimport sys\n\ntry:\n    from mcp.server import MCPServer  # MCP Python SDK v2\nexcept ImportError:\n    from mcp.server.mcpserver import MCPServer\n\napp = MCPServer("SwiftCartTrainingMCP")\nLOG_PATH = Path(__file__).with_name("section5_strands_mcp_tool_calls.jsonl")\n\nCATALOG = {\n    "SKU-101": {"item": "Reusable Bottle", "stock": 12, "price_inr": 499},\n    "SKU-102": {"item": "Desk Lamp", "stock": 0, "price_inr": 1299},\n}\nBASE_FEE = {"Mumbai": 40, "Ahmedabad": 65, "Delhi": 80}\n\ndef _record(name: str, arguments: dict, result: dict):\n    entry = {"tool": name, "arguments": arguments, "result": result}\n    with LOG_PATH.open("a", encoding="utf-8") as f:\n        f.write(json.dumps(entry, ensure_ascii=False) + "\\n")\n\ndef lookup_inventory(sku: str) -> dict:\n    sku = sku.upper().strip()\n    item = CATALOG.get(sku)\n    if item is None:\n        return {"sku": sku, "found": False, "available": False, "stock": 0}\n    return {"sku": sku, "found": True, "available": item["stock"] > 0, **item}\n\ndef calculate_shipping(city: str, weight_kg: float) -> dict:\n    if city not in BASE_FEE:\n        return {"city": city, "valid": False, "reason": "City not supported in training data"}\n    if not (0 < weight_kg <= 20):\n        return {"city": city, "valid": False, "reason": "Weight must be between 0 and 20 kg"}\n    return {"city": city, "valid": True, "weight_kg": weight_kg,\n            "shipping_cost_inr": round(BASE_FEE[city] + 12 * weight_kg, 2)}\n\n@app.tool()\ndef check_inventory(sku: str) -> dict:\n    """Check fictional shop inventory for an item SKU (for example SKU-101)."""\n    result = lookup_inventory(sku)\n    _record("check_inventory", {"sku": sku}, result)\n    return result\n\n@app.tool()\ndef estimate_shipping(city: str, weight_kg: float) -> dict:\n    """Calculate a fictional shipping charge for a city and weight in kg."""\n    result = calculate_shipping(city, weight_kg)\n    _record("estimate_shipping", {"city": city, "weight_kg": weight_kg}, result)\n    return result\n\n@app.resource("shop://catalog")\ndef read_catalog() -> str:\n    """Training-only fictional shop catalog."""\n    return json.dumps(CATALOG, ensure_ascii=False)\n\n@app.prompt()\ndef shop_helper() -> str:\n    """Prompt template for fictional inventory and shipping assistance."""\n    return "Use MCP inventory and shipping tools. This shop\'s data is fictional training data."\n\nif __name__ == "__main__":\n    if "--self-test" in sys.argv:\n        assert lookup_inventory("SKU-101")["stock"] == 12\n        assert lookup_inventory("SKU-102")["available"] is False\n        assert calculate_shipping("Mumbai", 2)["shipping_cost_inr"] == 64.0\n        print("PASS: deterministic inventory and shipping checks")\n    else:\n        app.run(transport="stdio")\n'
SERVER_PATH = (Path.cwd() / "mcp_shop_server.py").resolve()
ast.parse(SERVER_SOURCE)
SERVER_PATH.write_text(SERVER_SOURCE, encoding="utf-8")
LOG_PATH = SERVER_PATH.with_name("section5_strands_mcp_tool_calls.jsonl")
print("MCP server file:", SERVER_PATH.name)
print("PASS: local MCP server script created")

### Cell 4 — Verify the server's business rules

The server is run briefly in a special `--self-test` mode. This only checks the fictional calculations. It does not call the LLM or MCP yet.

In [ ]:
# Cell 4 — Smoke test the server file using the active Python kernel
import subprocess, sys
result = subprocess.run([sys.executable, str(SERVER_PATH), "--self-test"],
                        text=True, capture_output=True, timeout=45)
print(result.stdout.strip())
assert result.returncode == 0, result.stderr[-2000:]
assert "PASS:" in result.stdout
print("PASS: MCP server script and Python rules verified")

### Cell 5 — Real MCP Client ↔ Server protocol test

A client starts the server using **stdio**, discovers its Tools, Resource, and Prompt, then calls both Tools. This is actual MCP protocol traffic, not a Python function shortcut.

In [ ]:
# Cell 5 — MCP initialize, tool discovery and live protocol calls
import json, sys
from pathlib import Path

# Fix for Windows Jupyter: ipykernel stderr cannot be passed to subprocess.Popen.
# A genuine disk-file stream has a working fileno() for the child-process stderr.
MCP_STDERR_PATH = (Path.cwd() / "section5_mcp_stderr.log").resolve()
MCP_STDERR_STREAM = MCP_STDERR_PATH.open("a", encoding="utf-8")
assert MCP_STDERR_STREAM.fileno() >= 0
print("Windows/Jupyter MCP stderr redirected to:", MCP_STDERR_PATH.name)

def mcp_server_params():
    return StdioServerParameters(command=sys.executable, args=[str(SERVER_PATH)])

async def check_real_mcp_protocol():
    async with stdio_client(mcp_server_params(), errlog=MCP_STDERR_STREAM) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            names = [t.name for t in tools.tools]
            print("Tools discovered:", names)
            assert {"check_inventory", "estimate_shipping"}.issubset(set(names))
            resources = await session.list_resources()
            prompts = await session.list_prompts()
            print("Resources:", [str(r.uri) for r in resources.resources])
            print("Prompts:", [p.name for p in prompts.prompts])
            a = await session.call_tool("check_inventory", arguments={"sku": "SKU-101"})
            b = await session.call_tool("estimate_shipping", arguments={"city": "Mumbai", "weight_kg": 2.0})
            # SDK v2 CallToolResult uses the snake_case attribute is_error.
            # This is a protocol result status; no AWS or paid API is used.
            print("Inventory tool error:", a.is_error)
            print("Shipping tool error:", b.is_error)
            assert not a.is_error and not b.is_error, (a.model_dump(), b.model_dump())
            assert "SKU-101" in str(a.model_dump())
            assert "64" in str(b.model_dump())
            return names

mcp_tool_names = await check_real_mcp_protocol()  # Jupyter supports top-level await
print("PASS: Real MCP stdio discovery + both tool calls succeeded")

### Cell 6 — Configure Strands Agent with Ollama + MCP

This creates the local model and MCP client factory. The tool-using AI request is in the next cell. This step alone does **not** prove that the LLM selected a tool.

In [ ]:
# Cell 6 — Build local Strands components; still no paid API
from strands import Agent
from strands.models.ollama import OllamaModel
from strands.tools.mcp import MCPClient

model = OllamaModel(host=OLLAMA_HOST, model_id=OLLAMA_MODEL)

def make_mcp_client():
    return MCPClient(lambda: stdio_client(mcp_server_params(), errlog=MCP_STDERR_STREAM))

print("PASS: Local Strands Ollama model and MCP client factory configured")

### Cell 7 — Ask the Strands Agent to call **both** MCP Tools

This is the real model invocation on your laptop. The server logs actual calls, and Python checks whether the agent used **both** tools. The `qwen3:1.7b` model is small and may fail to use tools consistently; in that case the cell prints **PARTIAL**, not a false PASS. You can rerun Cell 7 once, or later try a stronger local model if your hardware permits.

In [ ]:
# Cell 7 — Real LLM-driven MCP tool use, verified by server logs
import json
if LOG_PATH.exists():
    LOG_PATH.unlink()  # Clear the direct calls from Cell 5; count only agent-initiated calls

agent_response_text = ""
agent_error = None
try:
    with make_mcp_client() as client:
        agent_tools = client.list_tools_sync()
        assert agent_tools, "Strands could not discover MCP tools"
        agent = Agent(
            model=model,
            tools=agent_tools,
            system_prompt=(
                "You are a careful assistant for a FICTIONAL practice store. "
                "Before answering, call the MCP tool check_inventory for SKU-101 "
                "AND call the MCP tool estimate_shipping for Mumbai and weight_kg=2.0. "
                "Use returned tool facts exactly; never invent numbers. "
                "Answer briefly in Hindi."
            ),
            callback_handler=None,
        )
        output = agent(
            "For training order SKU-101, check current fictional stock, then estimate "
            "shipping to Mumbai for 2.0 kg. Use BOTH MCP tools and show the facts in Hindi."
        )
        agent_response_text = str(output)
        print("=== STRANDS AGENT REPLY ===")
        print(agent_response_text[:5000])
except Exception as error:
    agent_error = f"{type(error).__name__}: {str(error)[:500]}"
    print("Agent call did not finish:", agent_error)

actual_calls = []
if LOG_PATH.exists():
    for line in LOG_PATH.read_text(encoding="utf-8").splitlines():
        if line.strip():
            actual_calls.append(json.loads(line))
called_names = [item["tool"] for item in actual_calls]
print("MCP tool calls observed:", called_names)
strands_mcp_pass = {"check_inventory", "estimate_shipping"}.issubset(set(called_names)) and bool(agent_response_text.strip())
if strands_mcp_pass:
    print("PASS: Strands Agent invoked BOTH real MCP tools using local Qwen3")
else:
    print("PARTIAL: MCP protocol works (Cell 5), but full autonomous Strands tool use is not verified")
    print("Try running Cell 7 once again; small local models sometimes omit tool calls.")

### Cell 8 — Save the result honestly

Saves a Markdown report plus a JSON evidence file, including exactly which tools the Strands LLM used. A saved file is **not** proof of a PASS by itself; it records `complete: true/false`.

In [ ]:
# Cell 8 — Save report + actual tool-call evidence
from pathlib import Path
import json

evidence = {
    "section": 5,
    "practical": "Strands Agent + MCP Tools + local Ollama",
    "model": OLLAMA_MODEL,
    "mcp_transport": "stdio local subprocess",
    "protocol_tool_names": globals().get("mcp_tool_names", []),
    "actual_agent_tool_calls": actual_calls,
    "agent_response": agent_response_text,
    "agent_error": agent_error,
    "complete": strands_mcp_pass,
    "fictional_training_data": True,
}
json_path = Path("section5_strands_mcp_evidence.json")
md_path = Path("section5_strands_mcp_report.md")
json_path.write_text(json.dumps(evidence, ensure_ascii=False, indent=2), encoding="utf-8")
md_path.write_text(
    "# Section 5 — Strands + MCP (Ollama)\\n\\n".replace("\\n", "\n")
    + ("**Status: PASS — both tools called**\n\n" if strands_mcp_pass else "**Status: PARTIAL — inspect evidence**\n\n")
    + "## Agent reply\n\n" + (agent_response_text or "No agent reply") + "\n\n"
    + "## Tools observed\n\n" + ", ".join(called_names) + "\n",
    encoding="utf-8",
)
print("Saved:", md_path.name, "and", json_path.name)
print("PASS: Practical 2 fully completed" if strands_mcp_pass else "SAVED: partial evidence — do not mark fully completed")

---
### What comes next?

After Cell 7 genuinely confirms both MCP tools and Cell 8 saves the evidence, continue to **Section 5 Practical 3 — A2A Travel Agents** (two agent servers and an orchestrator).

**Official references:**
- [Course: How Agents Talk](https://scaler-content.github.io/How-agents-talk/)
- [Strands — MCP Tools](https://strandsagents.com/docs/user-guide/sdk/tools/mcp-transports/)
- [Strands — Ollama local model](https://strandsagents.com/docs/user-guide/sdk/quickstart/python/)

**Caution:** The original instructor's MCP+Strands lab uses Amazon Bedrock and a v1 MCP SDK. This is a **local adaptation** using your existing Ollama model and current MCP SDK v2, not a byte-for-byte copy of the instructor's notebook.